# Exploring the Available Files

Now that we know a bit more about the ePIC software stack and the different stages involved, we will take a closer look at the simulation campaign files that are already available for us.

In addition to the material in this tutorial, you may want to consult the following resources -

- [The production working group webpage](https://eic.github.io/epic-prod/)
- [Further information on accessing the simulation campaign files](https://eic.github.io/resources/production_file.html)
    - Details on using *rucio* are included here, we are transitioning towards this being the default for file access

We will begin by setting up our server and importing some relevant packages -

## Setup

Run the cell below to import some packages we'll need. We also set our server name here and open our file system. Once you run this cell, I recommend minimising this subsection.

In [ ]:
import os
from XRootD import client
# See - https://pythonhosted.org/xrootdfs/ - For some details on available commands
# Create XRootD client
eic_server = 'root://dtn-eic.jlab.org/'
fs = client.FileSystem(eic_server)

## Folder Structure

The simulation campaign files stored on the xrootd server are sorted in a fairly logical manner. In this section, we will explore this folder structure and take a look at how we can browse the files stored on this server.

### Exploring the Structure

To begin with, files relating to ePIC are stored under - 

- /volatile/eic/EPIC/

We can take a look at what is in this directory -

In [ ]:
status, files = fs.dirlist("/volatile/eic/EPIC")
for entry in files:
    print(entry.name)

So, a few different folders. There are three we will focus on though -

- EVGEN: The input hepmc3 datasets
    - E.g. some files that have been supplied by a physics event generator 
- FULL: The full GEANT4 output root files (usually only saved for a fraction of runs)
    - If running a simulation yourself, this would be your output from processing npsim
- RECO: The output root files from the reconstruction
    - And again, if running yourself, this would be your output from EICrecon
 
Let's take a look in RECO

In [ ]:
status, files = fs.dirlist("/volatile/eic/EPIC/RECO")
for entry in files:
    print(entry.name)

This is a list of different simulation campaigns. These are formatted as -

    - YY.MM.Campaign

So 25.08.0 is the March 2025 simulation campaign. Some files will not be run in .0 campaigns, see [the simulation campaign pages](https://eic.github.io/epic-prod/documentation/default_datasets.html) for specifics.

For now, we'll jump into 25.03.1 -

In [ ]:
status, files = fs.dirlist("/volatile/eic/EPIC/RECO/25.03.1")
for entry in files:
    print(entry.name)

Now there's just one directory, this represents the detector geometry that was used for this campaign. In this case, epic_craterlake. What's in here?

In [ ]:
status, files = fs.dirlist("/volatile/eic/EPIC/RECO/25.03.1/epic_craterlake")
for entry in files:
    print(entry.name)

We now have distinct categories of physics simulations that have been processed -

- BACKGROUNDS: Background events/reactions
- SINGLE: Single particle simulations (typically to check specific detector response to a specific particle)
- SIDIS: Semi-Inclusive Deep Inelastic Scattering
- DIS: Deep Inelastic Scattering
- EXCLUSIVE: Exclusive physics processes/reactions

The file we downloaded previously was a DIS reaction, so let's jump in there - 

In [ ]:
status, files = fs.dirlist("/volatile/eic/EPIC/RECO/25.03.1/epic_craterlake/DIS")
for entry in files:
    print(entry.name)

Just three directories -

- CC: Charged current
- NC: Neutral current
- BeAGLE1.03.02-1.0: A specific event genearator (BeAGLE) version

Again, we had NC previously so let's follow that -

In [ ]:
status, files = fs.dirlist("/volatile/eic/EPIC/RECO/25.03.1/epic_craterlake/DIS/NC")
for entry in files:
    print(entry.name)

Again, three directories. This time, nice and straightforward, these are just the beam energy combinations that were processed. Typically, these are expressed as -

- ElectronBeamEnergy x Proton(Ion) Beam Energy
    - 18x275
        - 18 GeV electrons on 275 GeV protons
    - 10x100
        - 10 GeV electrons on 100 GeV protons
    - 5x41
        - 5 GeV electrons on 41 GeV protons
     
And again, we previously downloaded an 18x275 file, so let's proceed with that -

In [ ]:
status, files = fs.dirlist("/volatile/eic/EPIC/RECO/25.03.1/epic_craterlake/DIS/NC/18x275")
for entry in files:
    print(entry.name)

4 choices, this time, these correspond to the minimum Q2 value (4-monentum exchange squared) in the simulated files. Let's check that, this time though we'll only print the first 10 entries -

In [ ]:
status, files = fs.dirlist("/volatile/eic/EPIC/RECO/25.03.1/epic_craterlake/DIS/NC/18x275/minQ2=10")
i=0
for entry in files:
    i+=1
    print(entry.name)
    if i > 10:
        break

print("\nThere are:",files.size," files in total in directory - /volatile/eic/EPIC/RECO/25.03.1/epic_craterlake/DIS/NC/18x275/minQ2=10\n")

Slightly cumbersome, but if we check the number of files here, we'll find there are 9000. So unless we want all of those printed to screen, that's unfortunately what we have to do. 

You might also notice that the numbering at the end of our files looks slightly odd, it isn't sequential. We could make this look a bit nicer and print things out slightly differently using something like the following though -

In [ ]:
status, files = fs.dirlist("/volatile/eic/EPIC/RECO/25.03.1/epic_craterlake/DIS/NC/18x275/minQ2=10/")
flist=[]
for entry in files:
    flist.append(entry.name) # Add all of our entries to a new list, flist
    
for entry in sorted(flist)[0:10]: # Sort our list and slice off the first 10 entries, print all of these
    print(entry)

Note, now that we have this as a nice, sorted list, we could do more with it. For example, if we had our opening file path as a variable, we could quite quickly write something that copies the first 10 files of a certain process for example. Or maybe we want to check the size of them?

We could check the size of the first 10 entries using -

In [ ]:
TotalSize=0
N=10
for entry in sorted(flist)[0:N]:
    fname = "/volatile/eic/EPIC/RECO/25.03.1/epic_craterlake/DIS/NC/18x275/minQ2=10/" + entry
    print(entry," File Size: ",(((fs.stat(fname))[1]).size)/(1024**2), "MB")
    TotalSize+=(((fs.stat(fname))[1]).size)/(1024**2)
print("\nSize of all ", N, " files is: ", TotalSize, " MB\n")

### Summary/Overview

So, let's summarise and then move to an exercise. Our final file path for a reconstruction output file is -

- /volatile/eic/EPIC/RECO/25.03.1/epic_craterlake/DIS/NC/18x275/minQ2=10/pythia8NCDIS_18x275_minQ2=10_beamEffects_xAngle=-0.025_hiDiv_1.0000.eicrecon.edm4eic.root

Let's generalise this -

- /volatile/eic/EPIC/OUTPUT_TYPE/CAMPAGIN/DETECTOR_GEOMETRY/PHYSICS_PROCESS_TYPE/PHYSICS_PROCESS/BEAM_ENERGY/OTHER_CONDITIONS/FILE.eicrecon.edm4eic.root
    - OUTPUT_TYPE - Is this event generator, simulation or reconstruction output?
    - CAMPAIGN - Which simulation campaign is this from?
    - DETECTOR_GEOMETRY - Which detector geometry/design was used to run this simulation?
    - PHYSICS_PROCESS_TYPE - Broadly, what category of physics process is this?
    - PHYSICS_PROCESS - What physics process is this?
    - BEAM_ENERGY - What beam energy combination was used?
    - OTHER_CONDITIONS - Other simulation conditions, such as a range for a specific kinematic variable (Q2, t or similar)

Some directories may include event generator versions after the physics process too. You may also find some examples where there are multiple sets of additional conditions.

## Exercise

1. Find the latest *reconstruction* output for Deep Exclusive Meson Production (DEMP) events from 10 GeV electrons on 130 GeV protons.
    - Output all of the sub directories after the point at which the beam energy is selected
2. Determine the total number of output files and the size of all of these output files, consider all kinematic conditions and reactions.
    - **Hint** - By default, the file size is returned in Bytes, will this be a useful unit for your output?
    - **Hint** - Try "*fs.stat(path)*", what is the output? What is the output if you provide a file as the path?